In [1]:
import sys
sys.path.append("../")
import neo4j
import pandas as pd
import pandas as pd
from utils_embeddings import load_embedding_model_std
from neo4j_graphrag.indexes import create_vector_index
from neo4j_graphrag.indexes import upsert_vectors
from neo4j_graphrag.types import EntityType
from qdrant_client import QdrantClient
from neo4j_graphrag.retrievers import QdrantNeo4jRetriever
import json

c:\Users\paul-\anaconda3\envs\py310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = load_embedding_model_std()

Loading embedding model: paraphrase-multilingual-MiniLM-L12-v2


In [3]:
model

SentenceTransformer(
  (0): Transformer({'max_seq_length': 128, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
)

In [4]:
NEO4J_URI = "bolt://localhost:7688"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = "busticket123"

driver = neo4j.GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

In [5]:
INDEX_NAME = "embedding-index"
DIMENSION=384

In [6]:
testEmb = model.encode("This is a test sentence.").tolist()

In [7]:
len(testEmb)

384

In [8]:
COLLECTION_NAME = "simplified_test"

In [11]:
# 1. Verbindungen herstellen
client = QdrantClient("localhost", port=6333)

print("1. Verbindungen hergestellt")

# 2. Collection erstellen
client.recreate_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "size": model.get_sentence_embedding_dimension(),
        "distance": "Cosine"
    }
)
print(f"2. Qdrant Collection erstellt")

# 3. Alte Testdaten in Neo4j löschen
with driver.session() as session:
    session.run("MATCH (t:POI) DETACH DELETE t")
    print("3. Alte Testdaten in Neo4j gelöscht")

# 4. Testdaten erstellen
pois = [
    {
        "id": 1,
        "name": "Hamburger Michel",
        "description": "Die St. Michaelis Kirche ist ein bekanntes Wahrzeichen von Hamburg",
        "tags": ["kirche", "sehenswürdigkeit", "hamburg"],
        "coordinates": {"lat": 53.5486, "lon": 9.9793}
    },
    {
        "id": 2,
        "name": "Elbphilharmonie",
        "description": "Konzerthaus und Wahrzeichen der Hamburger Hafencity",
        "tags": ["musik", "konzert", "sehenswürdigkeit", "hamburg"],
        "coordinates": {"lat": 53.5413, "lon": 9.9841}
    },
    {
        "id": 3, 
        "name": "Landungsbrücken",
        "description": "Historische Anlegestellen am Hamburger Hafen",
        "tags": ["hafen", "schiffe", "hamburg"],
        "coordinates": {"lat": 53.5462, "lon": 9.9669}
    },
    {
        "id": 4,
        "name": "St. Petri Kirche",
        "description": "Eine der fünf Hauptkirchen in Hamburg",
        "tags": ["kirche", "hamburg", "altstadt"],
        "coordinates": {"lat": 53.5497, "lon": 9.9975}
    },
    {
        "id": 5,
        "name": "Fischrestaurant Hafenkante",
        "description": "Beliebtes Fischrestaurant mit Blick auf den Hafen",
        "tags": ["restaurant", "fisch", "hamburg", "hafen"],
        "coordinates": {"lat": 53.5450, "lon": 9.9700}
    }
]

# 5. Knoten in Neo4j erstellen
with driver.session() as session:
    for poi in pois:
        # ID als String speichern (wichtig für Abgleich)
        poi_id_str = str(poi["id"])
        
        query = """
        CREATE (p:POI {
            id: $id,
            name: $name,
            description: $description,
            tags: $tags,
            latitude: $lat,
            longitude: $lon
        })
        RETURN p.id as stored_id
        """
        
        result = session.run(
            query, 
            id=poi_id_str,  # Als String speichern
            name=poi["name"],
            description=poi["description"],
            tags=poi["tags"],
            lat=poi["coordinates"]["lat"],
            lon=poi["coordinates"]["lon"]
        )
        
        # Prüfen, ob ID korrekt gespeichert wurde
        stored_id = result.single()["stored_id"]
        print(f"POI {poi['name']} gespeichert mit ID: {stored_id} (Typ: {type(stored_id).__name__})")

print(f"5. {len(pois)} Knoten in Neo4j erstellt")

# 6. Debug: Prüfen der gespeicherten Daten in Neo4j
with driver.session() as session:
    result = session.run("MATCH (p:POI) RETURN p.id AS id, p.name AS name")
    print("\nGespeicherte POIs in Neo4j:")
    for record in result:
        print(f"ID: {record['id']} (Typ: {type(record['id']).__name__}) - Name: {record['name']}")

# 7. Embeddings erstellen und in Qdrant speichern
for poi in pois:
    # Text für Embedding kombinieren
    combined_text = f"""name: {poi['name']}
Beschreibung: {poi['description']}
Schlagwörter: {', '.join(poi['tags'])}"""
    
    # Embedding erzeugen
    vector = model.encode(combined_text).tolist()
    
    # ID als Integer für Qdrant
    poi_id_int = poi["id"]
    
    # In Qdrant speichern
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=[{
            "id": poi_id_int,  # Integer ID für Qdrant
            "vector": vector,
            "payload": {
                "neo4j_id": str(poi_id_int),  # Als String im payload speichern
                "name": poi["name"]
            }
        }]
    )

print(f"7. {len(pois)} Embeddings in Qdrant gespeichert")

# 8. Debug: Prüfen der gespeicherten Punkte in Qdrant
collection_info = client.get_collection(collection_name=COLLECTION_NAME)
print(f"\nQdrant Collection '{COLLECTION_NAME}' enthält {collection_info.points_count} Vektoren")


search_texts = [
    "Wo kann ich Kirchen in Hamburg besichtigen?",
    "Welche Sehenswürdigkeiten gibt es am Hamburger Hafen?",
    "Ich suche ein Restaurant"
]
    

# Jede Suchanfrage testen
for search_text in search_texts:
    print(f"\nSuche nach: '{search_text}'")
    
    # Embedding erzeugen
    query_embedding = model.encode(search_text).tolist()
    
    # Suche durchführen
    try:

        # Direkter Check in Qdrant
        qdrant_results = client.search(
            collection_name=COLLECTION_NAME,
            query_vector=query_embedding,
            limit=2
        )
        
        if qdrant_results:
            print("\nDirekte Qdrant-Ergebnisse:")
            for i, res in enumerate(qdrant_results, 1):
                print(f"Qdrant-Ergebnis {i}:")
                print(f"  ID: {res.id} (Typ: {type(res.id).__name__})")
                print(f"  Payload: {res.payload}")
                print(f"  Score: {res.score:.4f}")
                
                # Neo4j-Knoten direkt abfragen
                with driver.session() as session:
                    neo4j_id = res.payload.get("neo4j_id")
                    if neo4j_id:
                        neo4j_result = session.run(
                            "MATCH (p:POI {id: $id}) RETURN p.name AS name",
                            id=neo4j_id
                        )
                        record = neo4j_result.single()
                        if record:
                            print(f"  Entsprechender Neo4j-Knoten gefunden: {record['name']}")
                        else:
                            print(f"  KEIN Neo4j-Knoten mit ID={neo4j_id} gefunden!")
    
    except Exception as e:
        print(f"Fehler bei der Suche: {str(e)}")



# 12. Aufräumen (optional)
print("\nBereinige Testdaten...")
with driver.session() as session:
    session.run("MATCH (p:POI) DETACH DELETE p")

driver.close()
print("Test abgeschlossen")

1. Verbindungen hergestellt
2. Qdrant Collection erstellt
3. Alte Testdaten in Neo4j gelöscht
POI Hamburger Michel gespeichert mit ID: 1 (Typ: str)
POI Elbphilharmonie gespeichert mit ID: 2 (Typ: str)
POI Landungsbrücken gespeichert mit ID: 3 (Typ: str)
POI St. Petri Kirche gespeichert mit ID: 4 (Typ: str)
POI Fischrestaurant Hafenkante gespeichert mit ID: 5 (Typ: str)
5. 5 Knoten in Neo4j erstellt

Gespeicherte POIs in Neo4j:
ID: 1 (Typ: str) - Name: Hamburger Michel
ID: 2 (Typ: str) - Name: Elbphilharmonie
ID: 3 (Typ: str) - Name: Landungsbrücken
ID: 4 (Typ: str) - Name: St. Petri Kirche
ID: 5 (Typ: str) - Name: Fischrestaurant Hafenkante


C:\Users\paul-\AppData\Local\Temp\ipykernel_7784\2876901718.py:7: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(
C:\Users\paul-\AppData\Local\Temp\ipykernel_7784\2876901718.py:17: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:
C:\Users\paul-\AppData\Local\Temp\ipykernel_7784\2876901718.py:61: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:
C:\Users\paul-\AppData\Local\Temp\ipykernel_7784\2876901718.py:95: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:


7. 5 Embeddings in Qdrant gespeichert

Qdrant Collection 'simplified_test' enthält 5 Vektoren

Suche nach: 'Wo kann ich Kirchen in Hamburg besichtigen?'

Direkte Qdrant-Ergebnisse:
Qdrant-Ergebnis 1:
  ID: 1 (Typ: int)
  Payload: {'neo4j_id': '1', 'name': 'Hamburger Michel'}
  Score: 0.7371
  Entsprechender Neo4j-Knoten gefunden: Hamburger Michel
Qdrant-Ergebnis 2:
  ID: 4 (Typ: int)
  Payload: {'neo4j_id': '4', 'name': 'St. Petri Kirche'}
  Score: 0.7158
  Entsprechender Neo4j-Knoten gefunden: St. Petri Kirche

Suche nach: 'Welche Sehenswürdigkeiten gibt es am Hamburger Hafen?'

Direkte Qdrant-Ergebnisse:
Qdrant-Ergebnis 1:
  ID: 5 (Typ: int)
  Payload: {'neo4j_id': '5', 'name': 'Fischrestaurant Hafenkante'}
  Score: 0.5337
  Entsprechender Neo4j-Knoten gefunden: Fischrestaurant Hafenkante
Qdrant-Ergebnis 2:
  ID: 1 (Typ: int)
  Payload: {'neo4j_id': '1', 'name': 'Hamburger Michel'}
  Score: 0.5020
  Entsprechender Neo4j-Knoten gefunden: Hamburger Michel

Suche nach: 'Ich suche ein Re

C:\Users\paul-\AppData\Local\Temp\ipykernel_7784\2876901718.py:152: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  qdrant_results = client.search(
C:\Users\paul-\AppData\Local\Temp\ipykernel_7784\2876901718.py:167: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:
C:\Users\paul-\AppData\Local\Temp\ipykernel_7784\2876901718.py:187: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with driver.session() as session:
